## Finetune RT-DETR (1 class, telltale)

This notebook downloads and unzips the main dataset (`telltale_one_class_fused.zip` from `estefoucher/sail-cv-telltales`), fuses it with your custom split dataset (1-class), downloads the 640 checkpoint, and runs a short finetuning (10–15 epochs).

**Enable GPU first (Colab):** **Runtime** → **Change runtime type** → set **Hardware accelerator** to **GPU** → **Save**. Then re-run the notebook from the start.

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## Install dependencies

In [ ]:
!pip install -q ultralytics huggingface_hub
from IPython import display
display.clear_output()

## Download and unzip main dataset from Hugging Face

Downloads `telltale_one_class_fused.zip` from `estefoucher/sail-cv-telltales` (1-class YOLO, train/val), then unzips it.

In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

main_data_dir = Path(HOME) / "main_data"
main_data_dir.mkdir(parents=True, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="estefoucher/sail-cv-telltales",
    filename="telltale_one_class_fused.zip",
    repo_type="dataset",
    local_dir=HOME,
    local_dir_use_symlinks=False,
)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(main_data_dir)
# Find dataset root: main_data_dir has train/val directly, or a subdir (e.g. telltale_one_class_fused) does
contents = [p for p in main_data_dir.iterdir() if p.is_dir()]
candidate = main_data_dir
for child in contents:
    if (child / "train").exists() or (child / "val").exists():
        candidate = child
        break
if not (candidate / "train").exists() and not (candidate / "val").exists():
    raise FileNotFoundError(f"Expected train/ or val/ under {main_data_dir}. Contents: {list(main_data_dir.iterdir())}")
main_data_dir = candidate.resolve()
print(f"Main dataset at: {main_data_dir}")
print("Contents:", list(main_data_dir.iterdir()))
for split in ("train", "val"):
    img_dir = main_data_dir / split / "images"
    n = len(list(img_dir.iterdir())) if img_dir.exists() else 0
    print(f"  {split}/images: {n} items")

## Upload your custom dataset (Colab)

If running on Colab: upload the zip of your **split + reduced** custom dataset (output of `split_yolo_by_source.py`). It must contain `train/`, `val/`, and `data.yaml` with `nc: 1`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()  # noqa: F821
    custom_zip = list(uploaded.keys())[0]
    import zipfile
    custom_data_dir = Path(HOME) / "custom_data"
    custom_data_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(custom_zip, 'r') as z:
        z.extractall(custom_data_dir)
    # Find dataset root: custom_data_dir has train/val directly, or a subdir (e.g. name_uploaded/) does
    contents = [p for p in custom_data_dir.iterdir() if p.is_dir()]
    for child in contents:
        if (child / "train").exists() or (child / "val").exists():
            custom_data_dir = child.resolve()
            break
    else:
        custom_data_dir = custom_data_dir.resolve()
    print(f"Custom dataset at: {custom_data_dir}")
except ImportError:
    custom_data_dir = Path(HOME) / "custom_data"  # set manually if not Colab
    print("Not on Colab: set custom_data_dir to your custom split dataset path.")
    print(f"Expected: {custom_data_dir} with train/, val/, data.yaml (nc=1)")

## Fuse main + custom dataset

In [ ]:
import shutil

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

def copy_split_fixed(src_base: Path, split: str, dst_img: Path, dst_lbl: Path, prefix: str) -> int:
    img_dir = src_base / split / "images"
    lbl_dir = src_base / split / "labels"
    if not img_dir.exists():
        return 0
    n = 0
    for f in img_dir.iterdir():
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            shutil.copy2(f, dst_img / f"{prefix}_{f.name}")
            n += 1
    if lbl_dir.exists():
        for f in lbl_dir.iterdir():
            if f.is_file() and f.suffix == ".txt":
                shutil.copy2(f, dst_lbl / f"{prefix}_{f.name}")
    return n

fused_dir = Path(HOME) / "fused_data"
fused_dir.mkdir(parents=True, exist_ok=True)
for split in ("train", "val"):
    (fused_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (fused_dir / split / "labels").mkdir(parents=True, exist_ok=True)

print("Main dataset at:", main_data_dir)
print("Custom dataset at:", custom_data_dir)
for split in ("train", "val"):
    dst_img = fused_dir / split / "images"
    dst_lbl = fused_dir / split / "labels"
    n_main = copy_split_fixed(main_data_dir, split, dst_img, dst_lbl, "main")
    n_custom = 0
    custom_path = Path(custom_data_dir)
    if custom_path.exists() and (custom_path / split).exists():
        n_custom = copy_split_fixed(custom_path, split, dst_img, dst_lbl, "custom")
    print(f"  {split}: {n_main} from main, {n_custom} from custom")

fused_yaml = fused_dir / "data.yaml"
fused_yaml.write_text(f"path: {fused_dir.resolve()}\ntrain: train/images\nval: val/images\nnc: 1\nnames: ['telltale']\n")
print(f"Fused dataset at: {fused_dir}")
for split in ("train", "val"):
    img_dir = fused_dir / split / "images"
    n = sum(1 for f in img_dir.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS)
    print(f"  {split}: {n} images total")

## Download 640 checkpoint from Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="estefoucher/tell-tale-detector",
    filename="weights/sailcv-rtdetrl640.pt",
    local_dir=HOME,
    local_dir_use_symlinks=False,
)
print(f"Checkpoint: {ckpt_path}")

## Train (finetune) 10–15 epochs

In [ ]:
from ultralytics import RTDETR

model = RTDETR(ckpt_path)

model.train(
    data=str(fused_yaml),
    epochs=12,
    patience=5,
    imgsz=640,
    batch=8,
    workers=4,
    project=os.path.join(HOME, "runs", "train"),
    name="finetune_rtdetr",
    augment=True,
)

## Output

Best weights are saved under `runs/train/finetune_rtdetr/weights/best.pt`. Download it and set `model_path` in your parameters YAML to use the finetuned detector.